# IR Practical 4: SMS Spam Classification using KNN and TF-IDF
**Aim:** Preprocess SMS messages, convert them to TF-IDF features, train a K-Nearest Neighbors classifier, evaluate it, and classify a new message.

The PDF uses an SMS spam dataset with `Msg` and `Label` columns.

### 1. Install machine-learning and text packages

Installs NLTK, scikit-learn, and pandas. They provide text preprocessing, classification algorithms and evaluation metrics, and tabular-data handling.

In [ ]:
%pip -q install nltk scikit-learn pandas

### 2. Import libraries

Imports pandas and NumPy, regular expressions, NLTK preprocessing tools, train/test splitting, TF-IDF vectorization, the K-nearest-neighbours classifier, and classification metrics.

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    confusion_matrix, classification_report
)

nltk.download("stopwords", quiet=True)

### 3. Load the SMS spam dataset

Reads the CSV dataset from its source URL into a DataFrame and prints a preview. Internet access is required for the download.

In [ ]:
# Dataset source represented in the PDF.
url = "https://raw.githubusercontent.com/junioralive/india-spam-sms-classification/main/dataset/spam.csv"

data = pd.read_csv(url)
print("Dataset loaded successfully!")
display(data.head())

### 4. Inspect dataset and labels

Displays the dataset shape, column names, and class distribution. This helps verify the file structure and understand the number of ham and spam examples.

In [ ]:
print("Shape:", data.shape)

print("\\nColumns:")
print(data.columns.tolist())

print("\\nClass Distribution:")
print(data.iloc[:, 1].value_counts())

### 5. Rename message and label columns

Renames the first two dataset columns to consistent names used by the rest of the notebook, then displays sample rows.

In [ ]:
# Rename the first two columns to the names used in the practical.
data = data.rename(columns={
    data.columns[0]: "message",
    data.columns[1]: "label"
})

print(data.head())

### 6. Prepare class labels

Normalizes the label text and maps ham to 0 and spam to 1. Rows without a recognized label or message are removed so the classifier receives valid examples.

In [ ]:
data["label"] = data["label"].astype(str).str.lower().str.strip()
data["label_num"] = data["label"].map({
    "ham": 0,
    "spam": 1
})

# Keep only rows with a recognized class label and a message.
data = data.dropna(subset=["message", "label_num"]).copy()
data["label_num"] = data["label_num"].astype(int)

print(data[["message", "label", "label_num"]].head())
print("\\nClass counts:")
print(data["label"].value_counts())

### 7. Define text preprocessing

Loads English stop words and creates a Porter stemmer. The preprocessing function lowercases the message, removes unwanted characters, tokenizes/filters words, removes stop words, and stems remaining terms.

In [ ]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def preprocess_text(text):
    text = str(text).lower()

    # Remove punctuation and digits while retaining letters and whitespace.
    text = re.sub(r"[^a-z\\s]", " ", text)

    # Tokenization
    words = text.split()

    # Stop-word removal
    words = [
        word for word in words
        if word not in stop_words
    ]

    # Stemming
    words = [
        stemmer.stem(word)
        for word in words
    ]

    return " ".join(words)

### 8. Clean all messages

Applies the preprocessing function to every message and prints an example before and after cleaning to show how the text is transformed.

In [ ]:
data["clean_message"] = data["message"].apply(preprocess_text)

print("Original:")
print(data["message"].iloc[0])

print("\\nAfter Preprocessing:")
print(data["clean_message"].iloc[0])

### 9. Separate features and target

Stores cleaned message text as X (the input feature) and numeric ham/spam labels as y (the target to predict).

In [ ]:
X = data["clean_message"]
y = data["label_num"]

print("Input samples:", len(X))
print("Output samples:", len(y))

### 10. Split training and testing data

Divides the dataset into training and test portions. stratify=y preserves the class proportions in both sets, and random_state=42 makes the split reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

### 11. Convert text to TF-IDF features

Fits TF-IDF on the training messages and transforms both training and test messages into numeric sparse vectors. TF-IDF gives higher weight to terms that are informative within a document and less weight to terms common across documents.

In [ ]:
vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training feature shape:", X_train_tfidf.shape)
print("Testing feature shape:", X_test_tfidf.shape)

### 12. Create the K-NN classifier

Initializes a K-nearest-neighbours classifier with five neighbours and Euclidean distance. It predicts a message's class based on the labels of nearby training vectors.

In [ ]:
knn = KNeighborsClassifier(
    n_neighbors=5,
    metric="euclidean"
)

print("K-NN model created successfully!")

### 13. Train the classifier

Fits the K-NN model using the TF-IDF training vectors and their known labels.

In [ ]:
knn.fit(X_train_tfidf, y_train)
print("K-NN model trained successfully!")

### 14. Predict test labels

Uses the trained classifier to predict whether each message in the held-out test set is ham or spam, then displays a sample of predictions.

In [ ]:
y_pred = knn.predict(X_test_tfidf)

print("Predictions:")
print(y_pred[:20])

### 15. Calculate accuracy

Computes the proportion of test examples classified correctly and prints it as both a fraction and a percentage.

In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("Accuracy (%):", accuracy * 100)

### 16. Calculate precision and recall

Precision measures how many messages predicted as spam are actually spam. Recall (sensitivity) measures how many actual spam messages were successfully identified.

In [ ]:
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print("Precision:", precision)
print("Recall/Sensitivity:", recall)

### 17. Create confusion matrix

Builds a confusion matrix comparing actual and predicted labels. It shows correct and incorrect predictions for each class.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

### 18. Print classification report

Displays precision, recall, F1-score, and support for the Ham and Spam classes, providing a more detailed evaluation than accuracy alone.

In [ ]:
print("Classification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Ham", "Spam"],
    zero_division=0
))

### 19. Test a new message

Prompts for a new SMS/email message, applies the same preprocessing and TF-IDF transformation, then predicts its class with the trained K-NN model.

In [ ]:
message = input("Enter an email/SMS message: ")

clean_message = preprocess_text(message)
message_vector = vectorizer.transform([clean_message])
prediction = knn.predict(message_vector)[0]

if prediction == 1:
    print("Prediction: SPAM")
else:
    print("Prediction: HAM / NOT SPAM")